# 赤方偏移の「種類」を比べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coxless/astro-sandbox/blob/main/cosmology/redshift_types.ipynb)

赤方偏移 $z$ は「光の波長がどれだけ伸びたか」で、$1 + z = \lambda_{\mathrm{obs}} / \lambda_{\mathrm{emit}}$ です。
ところが、観測した $z$ には、宇宙膨張のほかにもいろいろな運動や効果が混ざっています。

Pantheon+ のデータには、同じ超新星について 3 種類の赤方偏移が入っていました。

| 列 | 意味 |
| --- | --- |
| `zHEL` | 太陽系(の重心)から見た赤方偏移。観測そのもの(地球の自転・公転の分は補正済み) |
| `zCMB` | 宇宙マイクロ波背景放射(CMB)が一様に見える座標系から見た赤方偏移。太陽系の運動の分を取り除いたもの |
| `zHD` | さらに、超新星の母銀河の **固有運動**(宇宙膨張以外の運動)の推定値を取り除いたもの。ハッブル図にはこれを使う |

このノートブックでは

1. `zHEL` と `zCMB` の差から、**太陽系が CMB に対して動いている速さと向き** を求める
2. `zCMB` と `zHD` の差(固有運動)が、近くの超新星のハッブル図にどう効くかを見る
3. 近くの銀河では重力が膨張に勝つこと(アンドロメダ銀河の **青方偏移**)と、そこから局所銀河群の質量を見積もる
4. 赤方偏移を「ドップラー効果」と考えた場合と「空間の膨張」と考えた場合で、速度の意味がどう違うかを比べる
5. いろいろな効果の大きさを、速度に直して並べる

## 使うデータ

- **Pantheon+**(Scolnic et al. 2022, Brout et al. 2022): https://github.com/PantheonPlusSH0ES/DataRelease
  - リポジトリにライセンスの明記はありません。研究用に公開されたデータなので、使うときは論文を引用します。
- 近くの銀河の視線速度と距離: NED などに載っているおおよその値を、表として書き込んでいます

## 0. 準備

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.constants as const
from astropy.coordinates import SkyCoord
from astropy.cosmology import Planck18
from matplotlib.patches import Patch
from scipy.optimize import brentq

DATA_URL = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/"
    "Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES.dat"
)
C_KMS = 299_792.458  # 光速 [km/s]
H0 = 73.0  # [km/s/Mpc]。ここでは SH0ES の値を使う

raw = pd.read_csv(DATA_URL, sep=r"\s+")
# 同じ超新星の重複行は、赤方偏移と位置が同じなので 1 行だけ残す
sn = raw.drop_duplicates("CID").reset_index(drop=True)
print(f"超新星 {len(sn)} 個")
sn[["CID", "zHEL", "zCMB", "zHD", "VPEC", "RA", "DEC"]].head()

## 1. 太陽系は CMB に対して動いている: `zHEL` → `zCMB`

赤方偏移は掛け算で重なります。太陽系が速度 $\boldsymbol{v}_\odot$ で動いていると、方向 $\hat{\boldsymbol{n}}$ にある天体の赤方偏移は

$$
1 + z_{\mathrm{hel}} = (1 + z_{\mathrm{CMB}}) \left(1 - \frac{\boldsymbol{v}_\odot \cdot \hat{\boldsymbol{n}}}{c}\right)
$$

になります(進んでいる向きの天体は少し青方偏移し、反対側は赤方偏移する)。
つまり $\dfrac{1 + z_{\mathrm{CMB}}}{1 + z_{\mathrm{hel}}} - 1 \approx \dfrac{\boldsymbol{v}_\odot \cdot \hat{\boldsymbol{n}}}{c}$ で、
全天の超新星について並べれば、$\boldsymbol{v}_\odot$ の 3 成分を最小二乗法で決められます。

In [ ]:
coords = SkyCoord(sn["RA"].to_numpy() * u.deg, sn["DEC"].to_numpy() * u.deg)
n_hat = coords.cartesian.xyz.value.T  # 各超新星への単位ベクトル (N, 3)
dv = ((1 + sn["zCMB"]) / (1 + sn["zHEL"]) - 1).to_numpy() * C_KMS  # [km/s]

v_sun, *_ = np.linalg.lstsq(n_hat, dv, rcond=None)
apex = SkyCoord(*v_sun, representation_type="cartesian", frame="icrs")
speed = np.linalg.norm(v_sun)
print(f"太陽系の CMB に対する速さ: {speed:.1f} km/s")
print(f"向き(銀河座標): l = {apex.galactic.l.deg:.1f}°, b = {apex.galactic.b.deg:.1f}°  "
      f"(赤道座標: RA = {apex.spherical.lon.deg:.1f}°, Dec = {apex.spherical.lat.deg:.1f}°)")
print("Planck 2018 の CMB 双極子: 369.82 km/s, l = 264.02°, b = 48.25°")

In [ ]:
cos_angle = n_hat @ (v_sun / speed)
plt.figure(figsize=(7, 4))
plt.scatter(cos_angle, dv, s=4)
xx = np.linspace(-1, 1, 10)
plt.plot(xx, speed * xx, c="tab:red", label=f"v cos θ, v = {speed:.0f} km/s")
plt.xlabel("cos θ (angle from the apex of solar motion)")
plt.ylabel("c [(1+zCMB)/(1+zHEL) − 1] [km/s]")
plt.legend()
plt.show()

Pantheon+ がこの補正に使った CMB の双極子(Planck の観測)がそのまま戻ってきました。
CMB の温度は、この方向で約 0.1% 高く、反対側で 0.1% 低く見えます($\Delta T / T = v/c$)。

この 370 km/s は、太陽が天の川銀河の中を回る速さ(約 240 km/s)と、天の川銀河(局所銀河群)が CMB に対して動いている速さ(約 620 km/s)の合成です。
局所銀河群は、うみへび座・ケンタウルス座の方向にある大きな銀河の集まり(グレート・アトラクター、シャプレー超銀河団)に引っ張られていると考えられています。

## 2. 銀河の固有運動: `zCMB` → `zHD`

母銀河自身も、まわりの銀河や銀河団の重力に引かれて、宇宙膨張とは別に数百 km/s で動いています(固有運動、peculiar velocity)。
Pantheon+ では、近くの銀河の分布から重力を計算して、超新星ごとの固有運動 `VPEC` を推定し、

$$
1 + z_{\mathrm{CMB}} = (1 + z_{\mathrm{HD}}) \left(1 + \frac{v_{\mathrm{pec}}}{c}\right)
$$

で、膨張だけによる赤方偏移 $z_{\mathrm{HD}}$ を出しています。本当にこの関係になっているか確かめます。

In [ ]:
check = (1 + sn["zCMB"]) / (1 + sn["zHD"]) - 1 - sn["VPEC"] / C_KMS
print(f"関係式からのずれの最大値: {np.abs(check).max():.1e}(z の桁数の丸め程度)")
print(f"VPEC: 平均 {sn['VPEC'].mean():.0f} km/s, 標準偏差 {sn['VPEC'].std():.0f} km/s, 推定の誤差 VPECERR = {sn['VPECERR'].iloc[0]:.0f} km/s")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(sn["VPEC"], bins=50)
axes[0].set_xlabel("VPEC [km/s]")
axes[0].set_ylabel("number of SNe")
axes[1].scatter(sn["zHD"], sn["VPEC"], s=4)
axes[1].set_xscale("log")
axes[1].set_xlabel("zHD")
axes[1].set_ylabel("VPEC [km/s]")
plt.tight_layout()
plt.show()

### 固有運動が効くのは近くだけ

固有運動は数百 km/s 程度で、距離によらずほぼ同じ大きさです。一方、宇宙膨張による速さ $cz$ は遠いほど大きくなります。
そのため、固有運動による「距離のずれ」の割合は $\sigma_v / (cz)$ で、近くほど大きくなります。

超新星の明るさから出した距離($\mu$, SH0ES の較正を使った `MU_SH0ES`)と、赤方偏移から出した距離を比べて、
どちらの赤方偏移を使うとハッブル図のばらつきが小さくなるか見てみます。速度に直して比べると分かりやすいです。

$$
v_{\mathrm{res}} = cz - H_0\, d(\mu)
$$

In [ ]:
# 固有運動の影響が目立つ、ごく近く(cz < 4500 km/s)の超新星だけを使う
low = sn[(sn["zHD"] > 0.003) & (sn["zHD"] < 0.015) & (sn["IS_CALIBRATOR"] == 0)].copy()
d_mpc = 10 ** ((low["MU_SH0ES"] - 25) / 5)
# 近くなので、d_L と赤方偏移の関係は 1 次の補正だけ入れる(q0 = -0.55)
expect = H0 * d_mpc / (1 + (1 - (-0.55)) / 2 * low["zHD"])
for col in ["zHEL", "zCMB", "zHD"]:
    low["vres_" + col] = C_KMS * low[col] - expect
    print(f"{col:5s}: 速度の残差の標準偏差 = {low['vres_' + col].std():4.0f} km/s, 絶対値の中央値 = {low['vres_' + col].abs().median():4.0f} km/s")
print(f"(超新星 {len(low)} 個。明るさのばらつき 0.1 mag だけで、z = 0.01 では約 {C_KMS * 0.01 * 0.1 * np.log(10) / 5:.0f} km/s の残差になる)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for col, color in [("zHEL", "tab:gray"), ("zHD", "tab:blue")]:
    axes[0].scatter(low["zHD"], low["vres_" + col], s=6, c=color, alpha=0.6, label=col)
axes[0].axhline(0, c="k", lw=0.8)
axes[0].set_xlabel("zHD")
axes[0].set_ylabel("cz − H0 d [km/s]")
axes[0].set_ylim(-1200, 1200)
axes[0].legend()

zz = np.logspace(-3, 0, 100)
for sv in [150, 250, 400]:
    axes[1].plot(zz, 100 * sv / (C_KMS * zz), label=f"σ_v = {sv} km/s")
axes[1].axhline(100 * 0.1 * np.log(10) / 5, c="k", ls="--", label="SN Ia brightness scatter (0.1 mag)")
axes[1].axvline(0.023, c="gray", ls=":", label="z = 0.023 (SH0ES lower limit)")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("redshift z")
axes[1].set_ylabel("distance error from peculiar velocity [%]")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

- 左: `zHEL`(灰)→ `zHD`(青)で、残差は少し小さくなります。ただし差はそれほど大きくありません。
  超新星の明るさのばらつき(約 5%)が、この距離でも 100〜200 km/s に相当するからです。固有運動の推定 `VPEC` 自体も ±250 km/s の不確かさがあります
- 右: 固有運動 250 km/s による距離の誤差は、$z \approx 0.02$ で超新星そのものの明るさのばらつき(約 5%)と同じくらいになります。
  SH0ES がハッブル定数に $z > 0.023$ の超新星だけを使うのはこのためです

## 3. 近くの銀河: 重力が膨張に勝つ

もっと近く、数 Mpc 以内の銀河を見てみます。視線速度(太陽から見た値)と距離の、おおよその値を書き込みます。

In [ ]:
LOCAL = pd.DataFrame(
    [
        # name, v_hel [km/s], distance [Mpc], 銀河座標 l, b [deg]
        ("LMC", 262, 0.050, 280.47, -32.89),
        ("SMC", 146, 0.062, 302.80, -44.30),
        ("M31 (Andromeda)", -300, 0.77, 121.17, -21.57),
        ("M33 (Triangulum)", -179, 0.84, 133.61, -31.33),
        ("M81", -34, 3.6, 142.09, 40.90),
        ("Centaurus A", 547, 3.8, 309.52, 19.42),
        ("M101", 241, 6.9, 102.04, 59.77),
        ("M87 (Virgo cluster)", 1284, 16.5, 283.78, 74.49),
    ],
    columns=["name", "v_hel", "d_mpc", "l", "b"],
)

太陽は天の川銀河の中を約 240 km/s で回っているので、まずその分を取り除き、銀河の中心から見た速度 $v_{\mathrm{GSR}}$ にします(Schönrich et al. 2010 の太陽運動を使ったよく使われる式)。

$$
v_{\mathrm{GSR}} = v_{\mathrm{hel}} + 11.1\cos l \cos b + 251 \sin l \cos b + 7.25 \sin b \quad [\mathrm{km/s}]
$$

In [ ]:
l_rad, b_rad = np.radians(LOCAL["l"]), np.radians(LOCAL["b"])
LOCAL["v_gsr"] = LOCAL["v_hel"] + 11.1 * np.cos(l_rad) * np.cos(b_rad) + 251 * np.sin(l_rad) * np.cos(b_rad) + 7.25 * np.sin(b_rad)
LOCAL["v_hubble"] = H0 * LOCAL["d_mpc"]  # 宇宙膨張だけならこの速さで遠ざかるはず
LOCAL.round({"d_mpc": 3, "v_gsr": 0, "v_hubble": 0})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
dd = np.logspace(-2, 1.4, 100)
ax.plot(dd, H0 * dd, c="tab:red", label=f"Hubble flow v = {H0:.0f} d")
ax.scatter(LOCAL["d_mpc"], LOCAL["v_gsr"], c="tab:blue", zorder=3)
label_offset = {"Centaurus A": (-75, 5), "M101": (5, 5)}
for _, r in LOCAL.iterrows():
    ax.annotate(r["name"], (r["d_mpc"], r["v_gsr"]), textcoords="offset points",
                xytext=label_offset.get(r["name"], (5, -10)), fontsize=8)
ax.axhline(0, c="k", lw=0.8)
ax.set_xscale("log")
ax.set_xlim(0.03, 25)
ax.set_xlabel("distance [Mpc]")
ax.set_ylabel("velocity relative to the Galactic center [km/s]")
ax.legend(loc="upper left")
plt.show()

アンドロメダ銀河(M31)は、太陽から見ると −300 km/s、天の川銀河の中心から見ても約 −110 km/s で、**近づいて** きています(青方偏移)。
天の川銀河とアンドロメダ銀河は、お互いの重力で宇宙膨張から切り離され、数十億年後にぶつかると考えられています。
数 Mpc 程度まで離れると、ようやく宇宙膨張の線に近づきます(M81, M101 など。ただしばらつきはまだ大きい)。

### タイミング論法: アンドロメダの速度から局所銀河群の質量を量る

Kahn & Woltjer (1959) は、「天の川銀河とアンドロメダ銀河はビッグバン直後には同じ場所にいて、膨張で離れたあと、重力で引き戻されて今近づいている」と考えました。
2 つの銀河の距離 $r$ は、ケプラー運動の式(離心率 1 の楕円軌道)で

$$
r = A(1 - \cos\eta), \qquad t = B(\eta - \sin\eta), \qquad A^3 = G M B^2
$$

と書けます。今の距離 $r$、近づく速さ $v = dr/dt$、宇宙年齢 $t$ が分かれば、$\eta$ を解いて、2 つの銀河の質量の合計 $M$ が求まります。
$v t / r$ が $\eta$ だけの関数になるので、まずそれを解きます。

$$
\frac{v\,t}{r} = \frac{\sin\eta\,(\eta - \sin\eta)}{(1 - \cos\eta)^2}
$$

In [ ]:
m31 = LOCAL.set_index("name").loc["M31 (Andromeda)"]
r_now = m31["d_mpc"] * u.Mpc
v_now = m31["v_gsr"] * u.km / u.s  # 近づいているので負
t_now = Planck18.age(0)


def vt_over_r(eta):
    return np.sin(eta) * (eta - np.sin(eta)) / (1 - np.cos(eta)) ** 2


target = (v_now * t_now / r_now).decompose().value
eta = brentq(lambda e: vt_over_r(e) - target, np.pi + 1e-6, 2 * np.pi - 1e-6)  # 近づいている(π < η < 2π)
A = r_now / (1 - np.cos(eta))
B = t_now / (eta - np.sin(eta))
M_lg = (A ** 3 / (const.G * B ** 2)).to(u.Msun)

print(f"v t / r = {target:.3f}  →  η = {eta:.3f} rad")
print(f"最も離れていたときの距離 2A = {(2 * A).to(u.Mpc):.2f}")
print(f"天の川銀河 + アンドロメダ銀河の質量 ≈ {M_lg.value:.1e} 太陽質量")
t_collide = B * 2 * np.pi - t_now
print(f"(この単純なモデルで)衝突まで約 {t_collide.to(u.Gyr).value:.1f} Gyr")

約 $4 \times 10^{12}$ 太陽質量になります。星やガスとして見えている質量(合わせて $2 \times 10^{11}$ 程度)の 20 倍ほどで、
銀河のまわりに大量の **暗黒物質** があることの、古くからある証拠の 1 つです(最近の詳しい見積もりでも $3$〜$5 \times 10^{12}$ 太陽質量)。

> アンドロメダ銀河の横方向の速度(固有運動)は小さいことが HST や Gaia で分かっていて、ほぼ正面衝突のコースです。実際には、この単純なモデルよりもう少し先(約 45 億年後)に最初の接近をすると予想されています。

## 4. 「ドップラー効果」か「空間の膨張」か

遠くの銀河の赤方偏移を「遠ざかる速度によるドップラー効果」と考えると、$z$ が大きいときは特殊相対論の式を使うことになり、速度は光速を超えません。

$$
\text{非相対論: } v = cz, \qquad \text{特殊相対論: } \frac{v}{c} = \frac{(1+z)^2 - 1}{(1+z)^2 + 1}
$$

一般相対論の宇宙論では、赤方偏移は **光が飛んでいる間に空間が伸びた割合** そのものです。

$$
1 + z = \frac{a(t_{\mathrm{obs}})}{a(t_{\mathrm{emit}})}
$$

この見方での「遠ざかる速さ」は、ハッブル=ルメートルの法則 $v = H D$ で決まり、距離 $D$ が大きければ光速を超えます(空間の伸びなので相対論とは矛盾しません)。

- 今の遠ざかる速さ: $v_{\mathrm{now}} = H_0 D_C(z)$($D_C$: 今の固有距離 = 共動距離)
- 光を出したときの遠ざかる速さ: $v_{\mathrm{emit}} = H(z)\, D_C(z) / (1+z)$

Planck 2018 の ΛCDM で計算して比べます。

In [ ]:
cosmo = Planck18
zz = np.logspace(-2, np.log10(1100), 400)
v_linear = zz
v_sr = ((1 + zz) ** 2 - 1) / ((1 + zz) ** 2 + 1)
dc = cosmo.comoving_distance(zz)
v_now_c = (cosmo.H0 * dc / const.c).decompose().value
v_emit_c = (cosmo.H(zz) * dc / (1 + zz) / const.c).decompose().value

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(zz, v_linear, c="gray", ls=":", label="v = cz")
ax.plot(zz, v_sr, c="tab:green", label="special-relativistic Doppler")
ax.plot(zz, v_now_c, c="tab:blue", label="recession velocity now  H0 D(z)")
ax.plot(zz, v_emit_c, c="tab:orange", label="recession velocity at emission")
ax.axhline(1, c="k", lw=0.8)
z_sn_max = sn["zHD"].max()
ax.axvline(z_sn_max, c="tab:red", ls="--", lw=1, label=f"most distant SN in Pantheon+ (z = {z_sn_max:.2f})")
ax.axvline(1090, c="gray", ls="--", lw=1, label="CMB (z ≈ 1090)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(0.01, 10)
ax.set_xlabel("redshift z")
ax.set_ylabel("v / c")
ax.legend(fontsize=8)
plt.show()

z_c = brentq(lambda z: (cosmo.H0 * cosmo.comoving_distance(z) / const.c).decompose().value - 1, 0.5, 3)
print(f"今の遠ざかる速さが光速になる赤方偏移: z = {z_c:.2f}(距離 {cosmo.comoving_distance(z_c).to(u.Gpc):.2f})")
for z in [z_sn_max, 1090]:
    print(f"z = {z:7.2f}: 特殊相対論のドップラーなら v = {((1 + z) ** 2 - 1) / ((1 + z) ** 2 + 1):.3f} c, "
          f"今の遠ざかる速さ = {(cosmo.H0 * cosmo.comoving_distance(z) / const.c).decompose().value:.2f} c")

- $z \lesssim 0.1$ では、どの考え方でも $v \approx cz$ で違いはありません(ハッブル定数の測定はこの範囲)
- $z \approx 1.5$ より遠い銀河は、**今は光速より速く遠ざかって** います。それでも光が届くのは、光を出したあとに宇宙の膨張の速さ $H$ が下がってきたからです
- Pantheon+ の一番遠い超新星や CMB は、特殊相対論のドップラーで解釈すると光速の 0.8 倍・ほぼ 1 倍ですが、膨張の見方では今は光速の約 1.3 倍・約 3 倍の速さで遠ざかっています

どちらの解釈が正しいかは「速度」の定義の問題でもありますが、距離と赤方偏移の関係(光度距離)や、CMB の温度が $T \propto 1 + z$ で高くなっていることなど、観測はすべて一般相対論の膨張宇宙で一貫して説明されます。
遠くの超新星ほど明るさの変化がゆっくり見える **時間の伸び**($1 + z$ 倍)も観測されています(Goldhaber et al. 2001 など)。

## 5. いろいろな効果の大きさを並べる

赤方偏移(や青方偏移)を生む効果を、速度に直して並べてみます($z \approx v/c$)。

In [ ]:
G = const.G
c = const.c
grav_sun = (G * const.M_sun / (const.R_sun * c)).to(u.km / u.s).value  # 太陽表面の重力赤方偏移(速度換算)
# シリウス B: 質量 1.02 太陽質量、半径 0.0081 太陽半径
grav_wd = (G * 1.02 * const.M_sun / (0.0081 * const.R_sun * c)).to(u.km / u.s).value

EFFECTS = pd.DataFrame(
    [
        ("Earth rotation", 0.46, "motion"),
        ("Sun's surface gravity", grav_sun, "gravity"),
        ("Earth's orbit around the Sun", 29.8, "motion"),
        ("white dwarf gravity (Sirius B)", grav_wd, "gravity"),
        ("Sun around the Galaxy", 240, "motion"),
        ("galaxy peculiar velocity (typical)", 250, "motion"),
        ("Sun vs CMB (dipole)", speed, "motion"),
        ("Local Group vs CMB", 620, "motion"),
        ("Virgo cluster (expansion)", 1100, "expansion"),
        ("z = 0.023 (SH0ES limit)", C_KMS * 0.023, "expansion"),
        ("z = 0.1", C_KMS * 0.1, "expansion"),
        ("z = 1 (as cz)", C_KMS * 1.0, "expansion"),
    ],
    columns=["effect", "v_kms", "kind"],
)
colors = {"motion": "tab:blue", "gravity": "tab:purple", "expansion": "tab:red"}
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(EFFECTS["effect"], EFFECTS["v_kms"], color=EFFECTS["kind"].map(colors))
ax.set_xscale("log")
ax.set_xlabel("velocity equivalent c·Δz [km/s]")
ax.legend(handles=[Patch(color=color, label=kind) for kind, color in colors.items()], loc="lower right")
plt.tight_layout()
plt.show()
print(f"太陽表面の重力赤方偏移: {grav_sun:.3f} km/s, シリウス B: {grav_wd:.0f} km/s(観測値は 80 ± 5 km/s, Barstow et al. 2005)")

- 地球の自転・公転の分は、観測所で必ず補正されて `zHEL` になります
- 太陽系・銀河の運動(数百 km/s)は、$z \approx 0.01$(3000 km/s)だと 10% 前後にもなるので、近くの宇宙の測定では大問題です
- 重力赤方偏移は、白色矮星や銀河団の中心などでは測れる大きさですが、宇宙論的な距離の測定にはほぼ効きません

## 発展メモ

- 1 章の双極子を、$z < 0.05$ と $z > 0.05$ の超新星に分けて求めると何が変わるか(近い超新星には固有運動の「流れ」が乗っている)
- `VPEC` を使って、固有運動の全天の地図を描く(どの方向に銀河が流れているか)
- タイミング論法で、宇宙年齢を Planck の 138 億年から 1 章の値(約 125 億年)に変えると質量はどう変わるか

## 参考・クレジット

- Scolnic et al. (2022), ApJ 938, 113 / Brout et al. (2022), ApJ 938, 110(Pantheon+)
- Carr et al. (2022), *Pantheon+ analysis: Improving the redshifts and peculiar velocities of Type Ia supernovae used in cosmological analyses*, PASA 39, e046
- Peterson et al. (2022), *The Pantheon+ Analysis: Evaluating Peculiar Velocity Corrections in Cosmological Analyses with Nearby Type Ia Supernovae*, ApJ 938, 112
- Planck Collaboration (2020), *Planck 2018 results. I. Overview and the cosmological legacy of Planck*, A&A 641, A1(CMB 双極子)
- Schönrich, Binney & Dehnen (2010), *Local kinematics and the local standard of rest*, MNRAS 403, 1829
- Kahn & Woltjer (1959), *Intergalactic Matter and the Galaxy*, ApJ 130, 705(タイミング論法)
- Davis & Lineweaver (2004), *Expanding Confusion: Common Misconceptions of Cosmological Horizons and the Superluminal Expansion of the Universe*, PASA 21, 97
- Goldhaber et al. (2001), *Timescale Stretch Parameterization of Type Ia Supernova B-Band Light Curves*, ApJ 558, 359
- Barstow et al. (2005), *Hubble Space Telescope spectroscopy of the Balmer lines in Sirius B*, MNRAS 362, 1134
- 近くの銀河の視線速度・距離: NASA/IPAC Extragalactic Database (NED) などのおおよその値